# Load CSVs

In [1]:
import pandas as pd

In [4]:
inflation = pd.read_csv("inflation_clean.csv")
interest = pd.read_csv("interest_clean.csv")

# Inspect each dataset
## Double check they are standardize

In [5]:
print(inflation.columns)

Index(['date', 'country', 'inflation'], dtype='object')


In [6]:
print(interest.columns)

Index(['date', 'interest'], dtype='object')


In [22]:
print(inflation.duplicated(["date"]).sum())

201636


In [23]:
print(interest.duplicated(["date"]).sum())

0


## Convert interest in continuous monthly data (now it jumps in months where we dont have data)

In [31]:
interest["date"] = pd.to_datetime(interest["date"], errors="coerce")
interest["interest"] = pd.to_numeric(interest["interest"], errors="coerce")
interest = interest.sort_values("date").drop_duplicates(subset="date")

In [32]:
#Fill up each month for the inflation range

full_months = pd.date_range(
    start=inflation["date"].min(),
    end=inflation["date"].max(),
    freq="MS"   # Month Start
)

In [33]:
interest_monthly = (
    interest.set_index("date")
    .reindex(full_months)
    .rename_axis("date")
    .reset_index()
)

interest_monthly["interest"] = interest_monthly["interest"].ffill()

In [34]:
print(interest_monthly.head(15))

         date  interest
0  2015-01-01       NaN
1  2015-02-01       NaN
2  2015-03-01       NaN
3  2015-04-01       NaN
4  2015-05-01       NaN
5  2015-06-01       NaN
6  2015-07-01       NaN
7  2015-08-01       NaN
8  2015-09-01       NaN
9  2015-10-01       NaN
10 2015-11-01       NaN
11 2015-12-01      -0.3
12 2016-01-01      -0.3
13 2016-02-01      -0.3
14 2016-03-01      -0.4


In [35]:
print(interest_monthly.tail())

          date  interest
103 2023-08-01      3.75
104 2023-09-01      4.00
105 2023-10-01      4.00
106 2023-11-01      4.00
107 2023-12-01      4.00


In [36]:
print(interest_monthly.isna().sum())

date         0
interest    11
dtype: int64


### ECB interest rate data records policy change dates rather than a complete monthly series. 

To align it with monthly inflation data, we expanded the series to a monthly frequency and forward-filled values, assuming the policy rate remained constant until the next recorded change.

In [45]:
interest_monthly["interest"] = interest_monthly["interest"].ffill().bfill()

## Merge

In [46]:
df = inflation.merge(
    interest_monthly,
    on = "date",
    how = "left"
)

In [47]:
df.head()

,date,country,inflation,interest
0,2015-01-01,DE,-0.5,-0.3
1,2015-02-01,DE,-0.2,-0.3
2,2015-03-01,DE,0.3,-0.3
3,2015-04-01,DE,1.0,-0.3
4,2015-05-01,DE,1.6,-0.3


In [48]:
df.columns

Index(['date', 'country', 'inflation', 'interest'], dtype='object')

In [49]:
df.shape

(201744, 4)